> **GitHub管理版（2026-11-05移管）**
> 既存の実験コードを移管したものです。セル出力は除去しており、今回の移管では実験を再実行していません。入力データ・保存済み結果は従来のDrive等を参照します。実行条件、認証、保存先を確認してから実行してください。元ファイルと用途は [ノートブック一覧](https://github.com/ryy1210/RMT_utils/blob/main/notebooks/README.md) を参照してください。

# Llama V：DE前後の正規化特異値分布
対象は `model.layers.2.self_attn.v_proj`（第3ブロック）。保存済みLlama-3.2-3Bのスペクトルを使用し、モデルのロード・SVD・DEの再計算は行いません。CPUで実行できます。

VS CodeでPythonカーネルを選び「すべて実行」。必要なパッケージは numpy / pandas / matplotlib。
保存先は本ノートブックと同じポスター用フォルダ内の `generated_esd`。既存のポスター画像・TeXは上書きしません。

**尺度について**：保存配列は特異値そのものではなく $\lambda_i=s_i^2/n$。図では $x_i=\sqrt{\lambda_i}=s_i/\sqrt n$ を使います。DE後は $s_i(AWB)/\sqrt n$ です。同じ $\sqrt n$ で割っても、DEのスケーリングにより前後の横軸の値は異なります。分散まで揃える $s_i/\sqrt{n\sigma^2}$ とは異なります。

現在のポスター画像の尺度は既にこの正規化と整合的ですが、元の作図コードは未特定です。古い図のKS（0.129, 0.092）をそのまま再現するノートブックではありません。対象の位置・灰色ヒストグラム・黒破線・前後2パネルの構成を維持し、保存済み実験に基づく値を使用します。

In [ ]:
from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ローカルのGoogle Driveを自動検出。候補が複数ならここを明示的に指定。
roots = list((Path.home()/'Library/CloudStorage').glob('GoogleDrive-*/*/TUS/hashiguchi'))
if len(roots) != 1:
    raise RuntimeError('ROOTをTUS/hashiguchiの絶対パスに設定してください。')
ROOT = roots[0]
SOURCE = ROOT/'data/bulk_ks/llama/bulk-ks-20260925-v1-full'
TARGET = 'model.layers.2.self_attn.v_proj'
OUTPUT = ROOT/'IBIS2026/poster3_5/generated_esd'
POST_MP = 'fixed1'  # ポスターの説明に合わせ分散1。'bema'も選択可能。
BINS = 50
plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'legend.fontsize': 9,
                     'pdf.fonttype': 42, 'ps.fonttype': 42})
print('CPUのみ / モデル・APIキー不要')
print('保存先:', OUTPUT)


In [ ]:
table_path = SOURCE/'bulk_ks.csv'
frame = pd.read_csv(table_path)
rows = frame.loc[frame.name.eq(TARGET)]
assert len(rows) == 1, '対象行列の行が一意でありません'
row = rows.iloc[0]
spectrum_path = (SOURCE/row.spectrum_file).resolve()
assert spectrum_path.is_relative_to(SOURCE.resolve())
with np.load(spectrum_path, allow_pickle=False) as z:
    assert str(z['layer_name'].item()) == TARGET
    p, n = int(z['p']), int(z['n'])
    spectra = {key: np.asarray(z[key], dtype=float) for key in ('pre','post')}
assert (p,n) == (int(row.p),int(row.n))
assert 0 < p <= n
for a in spectra.values():
    assert len(a) == p and np.isfinite(a).all() and (a >= 0).all()
q = p/n
assert POST_MP in ('fixed1','bema')
variances = {'pre':float(row.sigma2_preDE),
             'post':1.0 if POST_MP == 'fixed1' else float(row.sigma2_postDE)}
ks_columns = {'pre':'KS_preDE', 'post':'KS_postDE_1' if POST_MP == 'fixed1' else 'KS_postDE'}
# 重要：配列はすでにs²/n。ここでさらにnで割らない。
singular_values = {key:np.sqrt(a) for key,a in spectra.items()}
summary = pd.DataFrame([{'condition':key,'p':p,'n':n,'q':q,
    'MP_variance':variances[key], 'full_KS_saved_current_method':float(row[ks_columns[key]]),
    'min_s_over_sqrt_n':float(x.min()), 'max_s_over_sqrt_n':float(x.max())}
    for key,x in singular_values.items()])
display(summary)


## 密度と図
MPの固有値密度を特異値の尺度へ変換した密度を使用：
$$g(x)=\frac{\sqrt{(b^2-x^2)(x^2-a^2)}}{\pi q\sigma^2 x},\quad a=\sigma(1-\sqrt q),\ b=\sigma(1+\sqrt q).$$
全特異値を用いて面積1のヒストグラムを作成。外れ値の切り捨てやバルクだけの再正規化は行いません。
KSは保存済みの現行方式による全体KSです。経験分布とMPを同じ単調変換で固有値から特異値へ変換してもKSは不変です。

In [ ]:
def mp_singular_density(x, variance):
    sigma = np.sqrt(variance)
    a,b = sigma*(1-np.sqrt(q)), sigma*(1+np.sqrt(q))
    y = np.zeros_like(x)
    inside = (x > a) & (x < b) & (x > 0)
    t = x[inside]
    y[inside] = np.sqrt((b*b-t*t)*(t*t-a*a))/(np.pi*q*variance*t)
    return y

def draw(ax, key, title=None):
    x = singular_values[key]
    upper = np.sqrt(variances[key])*(1+np.sqrt(q))
    xmax = max(x.max(),upper)*1.03
    bins = np.linspace(0,xmax,BINS+1)
    heights,edges,_ = ax.hist(x,bins=bins,density=True,color='0.55',
                             edgecolor='white',linewidth=.35,label='Empirical Density')
    assert np.isclose(np.sum(heights*np.diff(edges)),1)
    grid = np.linspace(0,xmax,4000)
    ax.plot(grid,mp_singular_density(grid,variances[key]),'k--',lw=1.2,label='MP Density')
    ax.set(xlim=(0,xmax),ylim=(0,None),ylabel='Probability Density')
    ax.set_xlabel(r'$s_i(W)/\sqrt{n}$' if key=='pre' else r'$s_i(AWB)/\sqrt{n}$')
    ax.ticklabel_format(axis='x',style='sci',scilimits=(-2,3),useMathText=True)
    ax.legend(loc='upper right')
    if title:
        ax.set_title(f'{title}   KS = {float(row[ks_columns[key]]):.3f}')

# 前後の横軸は独立。DEによる分散の変化を隠さない。
fig,axes = plt.subplots(1,2,figsize=(12.8,4.8),layout='constrained')
for ax,key,title in zip(axes,('pre','post'),('Before DE','After DE')):
    draw(ax,key,title)
plt.show()


In [ ]:
# PDF/PNGを新しい保存先へ出力。既存のimages/は変更しない。
OUTPUT.mkdir(parents=True,exist_ok=True)
fig.savefig(OUTPUT/'llama_v_2_normalized_pair.pdf',bbox_inches='tight')
fig.savefig(OUTPUT/'llama_v_2_normalized_pair.png',dpi=220,bbox_inches='tight')
for key,filename in [('pre','llama_v_2'),('post','llama_v_2_d')]:
    single,ax = plt.subplots(figsize=(6.4,4.8),layout='constrained')
    draw(ax,key)
    single.savefig(OUTPUT/(filename+'.pdf'),bbox_inches='tight')
    single.savefig(OUTPUT/(filename+'.png'),dpi=220,bbox_inches='tight')
    plt.close(single)
summary.to_csv(OUTPUT/'spectrum_summary.csv',index=False)
provenance = {'target':TARGET,'source_csv':str(table_path),'source_npz':str(spectrum_path),
    'spectrum_sha256':hashlib.sha256(spectrum_path.read_bytes()).hexdigest(),
    'csv_sha256':hashlib.sha256(table_path.read_bytes()).hexdigest(),
    'stored_quantity':'lambda=s**2/n','plotted_quantity':'sqrt(lambda)=s/sqrt(n)',
    'post_mp':POST_MP,'bins':BINS,'python':sys.version,
    'numpy':np.__version__,'pandas':pd.__version__}
(OUTPUT/'provenance.json').write_text(json.dumps(provenance,indent=2),encoding='utf-8')
print('保存完了:',OUTPUT)
print('ポスターに反映する際は、上の集計値に合わせてKS表記も更新してください。')
